# Catalog Metadata Search

This notebook explains how to configure and query the **collection metadata search backend** —
the system that powers `GET /catalogs/{catalog_id}/collections` with full-text search,
spatial filtering, and CQL2 expressions.

## Disambiguation

| Term | What it means |
|------|---------------|
| **metadata.override** | `RoutingPluginConfig` field — selects which `CollectionMetadataDriverProtocol` backend handles collection CRUD + search. Covered in this notebook. |
| **metadata.storage** | `RoutingPluginConfig` field — selects which *storage* drivers supply collection metadata at READ time (Iceberg table properties, DuckDB sidecars). See the *Collection Metadata Enrichment* notebook instead. |
| **Metadata enrichment** | Per-item enrichment via `DriverMetadataEnricher` — different from collection-level search. |

> **TL;DR** — This notebook is about *where collection records live and how to search them*.
> The *Collection Metadata Enrichment* notebook is about *where item-level metadata comes from at read time*.

## 1 — Drivers Available

Two metadata backends are built in:

| `driver_id` | Backend | Strengths |
|-------------|---------|----------|
| `postgresql_metadata` | PostgreSQL `catalog_metadata` table | Always available, transactional, CQL2 via SQL |
| `elasticsearch_metadata` | ES index `{index_prefix}_{catalog_id}` | Full-text, geo-shape, aggregations, multilanguage |

Use `GET /admin/drivers` to verify which drivers are registered and available in your deployment.

In [ ]:
import httpx
import os

BASE_URL = os.environ.get("DYNASTORE_BASE_URL") or "http://localhost:80"
CATALOG_ID = "my-catalog"
HEADERS = {"Authorization": "Bearer <your-token>"}

resp = httpx.get(f"{BASE_URL}/admin/drivers", headers=HEADERS)
resp.raise_for_status()

drivers = resp.json()["drivers"]
for driver_type, impls in drivers.items():
    if "metadata" in driver_type:
        print(f"\n{driver_type}")
        for d in impls:
            avail = "✓" if d["available"] else "✗"
            print(f"  {avail} {d['driver_id']}: {d['description'].get('en', '')}")

## 2 — Auto-Discovery (Default Behaviour)

Without explicit `metadata.override` config, the metadata router tries drivers in this order:

1. `elasticsearch_metadata` — if available and `on_failure=warn`, falls through
2. `postgresql_metadata` — always available fallback

This means **you get full-text + spatial search if ES is up, PG otherwise — with no config change required**.

## 3 — Explicit `metadata.override` Config

To pin the metadata backend for a catalog, set `metadata.override` in `CollectionRoutingConfig`.
This is useful when you want to guarantee ES is used (and fail fast if it is not available).

In [ ]:
routing_config = {
    "plugin_id": "CollectionRoutingConfig",
    "enabled": True,
    "operations": {
        "WRITE":  [{"driver_id": "ItemsPostgresqlDriver", "on_failure": "fatal"}],
        "READ":   [{"driver_id": "ItemsPostgresqlDriver"}],
        "SEARCH": [{"driver_id": "ItemsElasticsearchDriver", "on_failure": "warn"},
                   {"driver_id": "ItemsPostgresqlDriver"}]
    },
    "metadata": {
        "override": [
            {"driver_id": "CollectionElasticsearchDriver", "on_failure": "warn"},
            {"driver_id": "CatalogCorePostgresqlDriver"}
        ],
        "storage": []
    }
}

resp = httpx.post(
    f"{BASE_URL}/catalogs/{CATALOG_ID}/configs",
    json=routing_config,
    headers=HEADERS,
)
resp.raise_for_status()
print(resp.json())

## 4 — ES Metadata Driver Config (Immutable Index Prefix)

The ES metadata driver stores collections in a per-catalog index named
`{index_prefix}_{catalog_id}`. The `index_prefix` field is **immutable** —
it can be set once and never changed (to avoid orphaned indices).

Apply the driver config **before** applying the routing config above.

In [ ]:
es_meta_config = {
    "plugin_id": "driver:collection:metadata:elasticsearch",
    "index_prefix": "meta"   # index will be: meta_my-catalog
}

resp = httpx.post(
    f"{BASE_URL}/catalogs/{CATALOG_ID}/configs",
    json=es_meta_config,
    headers=HEADERS,
)
resp.raise_for_status()
print(resp.json())

## 5 — Multilanguage Search (`q`)

The `q` parameter searches all language values in `title` and `description` JSONB fields.
This works for both the PG driver (ILIKE across `jsonb_each_text`) and the ES driver
(multi-field `match` across language sub-fields).

In [ ]:
resp = httpx.get(
    f"{BASE_URL}/catalogs/{CATALOG_ID}/collections",
    params={"q": "temperature", "limit": 10},
    headers=HEADERS,
)
resp.raise_for_status()
result = resp.json()
print(f"Found {result['numberMatched']} collections matching 'temperature'")
for col in result["collections"]:
    print(f"  {col['id']}: {col.get('title', {}).get('en', '')}")

## 6 — Spatial Bounding Box Filter

The `bbox` parameter filters collections whose spatial extent intersects the given bounding box.
Requires `MetadataCapability.SPATIAL_FILTER` — supported by both PG and ES metadata drivers.

In [ ]:
resp = httpx.get(
    f"{BASE_URL}/catalogs/{CATALOG_ID}/collections",
    params={
        "bbox": "-180,-90,180,90",  # global extent
        "limit": 5,
    },
    headers=HEADERS,
)
resp.raise_for_status()
result = resp.json()
print(f"Found {result['numberMatched']} collections in bbox")

## 7 — CQL2 Filter

Advanced filtering via CQL2-JSON (requires `MetadataCapability.CQL_FILTER`).
The PG driver translates CQL2 to SQL via `pygeofilter`.
The ES driver translates to ES query DSL.

In [ ]:
cql2_filter = {
    "op": "and",
    "args": [
        {"op": "like", "args": [{"property": "title.en"}, "%climate%"]},
        {"op": "=", "args": [{"property": "license"}, "CC-BY-4.0"]}
    ]
}

resp = httpx.get(
    f"{BASE_URL}/catalogs/{CATALOG_ID}/collections",
    params={
        "filter": __import__('json').dumps(cql2_filter),
        "filter-lang": "cql2-json",
    },
    headers=HEADERS,
)
resp.raise_for_status()
print(resp.json())

## 8 — Aggregations

Faceted counts and statistics on metadata fields (requires `MetadataCapability.AGGREGATION`).
This is supported by the ES metadata driver; the PG driver returns basic counts.

In [ ]:
# Terms aggregation on 'license' field
resp = httpx.get(
    f"{BASE_URL}/catalogs/{CATALOG_ID}/collections/aggregate",
    params={"field": "license", "type": "terms"},
    headers=HEADERS,
)
if resp.status_code == 200:
    for bucket in resp.json().get("buckets", []):
        print(f"  {bucket['key']}: {bucket['doc_count']} collections")
else:
    print(f"Aggregation not supported by active driver: {resp.status_code}")

## 9 — PG Fallback

When ES is unavailable (connection error, timeout), the metadata router falls through
to the next entry in `metadata.override` — typically `postgresql_metadata`.

The PG driver supports:
- Full-text search via `ILIKE` across all language values (`MetadataCapability.SEARCH`)
- Spatial filter via PostGIS (`MetadataCapability.SPATIAL_FILTER`)
- Basic CQL2 via `pygeofilter` → SQL (`MetadataCapability.CQL_FILTER`)
- Soft delete (`MetadataCapability.SOFT_DELETE`)

No code change is needed to switch — the routing config `on_failure=warn` handles fallback automatically.

## 10 — RECORDS Collections + Null Geometry

RECORDS-type collections (tabular data without geometry, e.g. spreadsheets, CSVs) are
fully supported. A null geometry extent is STAC-compliant:

```json
{
  "extent": {
    "spatial": {"bbox": [[-180, -90, 180, 90]]},
    "temporal": {"interval": [[null, null]]}
  }
}
```

The ES metadata driver maps `extent.spatial.bbox` to a `geo_shape` field;
the PG driver stores it as JSONB. Both handle null geometry gracefully —
no geometry column is required for RECORDS collections.